In [ ]:
import numpy as np
from scipy.stats import gaussian_kde

# Convert lists to arrays if not already
total_distances = np.array(total_distances)
thicknesses = np.array(thicknesses)

# Create KDEs
kde_total = gaussian_kde(total_distances)
kde_thick = gaussian_kde(thicknesses)

# Create fine grid to evaluate KDEs
x_range = np.linspace(min(total_distances.min(), thicknesses.min()),
                      max(total_distances.max(), thicknesses.max()), 500)

# Evaluate KDEs
kde_total_values = kde_total(x_range)
kde_thick_values = kde_thick(x_range)

# Find peaks (x value where KDE is maximum)
peak_total = x_range[np.argmax(kde_total_values)]
peak_thick = x_range[np.argmax(kde_thick_values)]

print(f"Peak of total_distances: {peak_total:.2f} pixels")
print(f"Peak of thicknesses: {peak_thick:.2f} pixels")

In [1]:
def compute_hessian(J, x, y, dx=1.0, dy=1.0):
    """
    Compute 2D Hessian matrix at point (x,y) of array J using central differences.
    """
    d2f_dx2 = (J[y, (x+1)%J.shape[1]] - 2*J[y, x] + J[y, (x-1)%J.shape[1]]) / dx**2
    d2f_dy2 = (J[(y+1)%J.shape[0], x] - 2*J[y, x] + J[(y-1)%J.shape[0], x]) / dy**2
    d2f_dxdy = (
        J[(y+1)%J.shape[0], (x+1)%J.shape[1]] - J[(y+1)%J.shape[0], (x-1)%J.shape[1]]
        - J[(y-1)%J.shape[0], (x+1)%J.shape[1]] + J[(y-1)%J.shape[0], (x-1)%J.shape[1]]
    ) / (4*dx*dy)

    H = np.array([[d2f_dx2, d2f_dxdy],
                  [d2f_dxdy, d2f_dy2]])
    return H

def steepest_descent_direction(J, peaks, dx=1.0, dy=1.0):
    """
    For each peak, compute direction of steepest descent from Hessian.
    
    Parameters
    ----------
    J : 2D array
        J_parallel field
    peaks : list of (y,x) tuples
        Coordinates of detected local maxima
    dx, dy : float
        Grid spacing

    Returns
    -------
    directions : dict
        peak -> (eigenvalue, eigenvector)
    """
    directions = {}
    for (y, x) in peaks:
        H = compute_hessian(J, x, y, dx, dy)
        eigvals, eigvecs = np.linalg.eigh(H)  # sorted eigenvalues
        # largest magnitude eigenvalue
        idx = np.argmax(np.abs(eigvals))
        directions[(y, x)] = (eigvals[idx], eigvecs[:, idx])
    return directions

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

# -------------------------------
# 1️⃣ Hessian and steepest descent
# -------------------------------
def compute_hessian(J, x, y, dx=1.0, dy=1.0):
    """
    Compute 2D Hessian matrix at point (x,y) of array J using central differences.
    """
    d2f_dx2 = (J[y, (x+1)%J.shape[1]] - 2*J[y, x] + J[y, (x-1)%J.shape[1]]) / dx**2
    d2f_dy2 = (J[(y+1)%J.shape[0], x] - 2*J[y, x] + J[(y-1)%J.shape[0], x]) / dy**2
    d2f_dxdy = (
        J[(y+1)%J.shape[0], (x+1)%J.shape[1]] - J[(y+1)%J.shape[0], (x-1)%J.shape[1]]
        - J[(y-1)%J.shape[0], (x+1)%J.shape[1]] + J[(y-1)%J.shape[0], (x-1)%J.shape[1]]
    ) / (4*dx*dy)

    H = np.array([[d2f_dx2, d2f_dxdy],
                  [d2f_dxdy, d2f_dy2]])
    return H

def steepest_descent_direction(J, peaks, dx=1.0, dy=1.0):
    """
    For each peak, compute direction of steepest descent from Hessian.
    Returns a dict: peak -> (most negative eigenvalue, eigenvector)
    """
    directions = {}
    for (y, x) in peaks:
        H = compute_hessian(J, x, y, dx, dy)
        eigvals, eigvecs = np.linalg.eigh(H)
        idx = np.argmin(eigvals)  # most negative eigenvalue
        directions[(y, x)] = (eigvals[idx], eigvecs[:, idx])
    return directions

# -------------------------------
# 2️⃣ Get peaks
# -------------------------------
peaks = list(zip(*np.where(max_coords == 1)))

# -------------------------------
# 3️⃣ Compute steepest descent directions
# -------------------------------
directions = steepest_descent_direction(J_par, peaks, dx=0.125, dy=0.125)

# -------------------------------
# 4️⃣ Compute distances to connected region boundaries
# -------------------------------
def distance_to_edge(mask, x, y, dx, dy, step_size=0.5, max_steps=1000):
    """Compute distance from (x,y) along (dx,dy) inside mask until boundary."""
    # Forward
    xf, yf = x, y
    steps = 0
    while steps < max_steps:
        xf_new = xf + dx*step_size
        yf_new = yf + dy*step_size
        xi, yi = int(round(xf_new)), int(round(yf_new))
        if xi<0 or yi<0 or xi>=mask.shape[1] or yi>=mask.shape[0]:
            break
        if not mask[yi, xi]:
            break
        xf, yf = xf_new, yf_new
        steps += 1
    dist_forward = np.hypot(xf - x, yf - y)

    # Backward
    xb, yb = x, y
    steps = 0
    while steps < max_steps:
        xb_new = xb - dx*step_size
        yb_new = yb - dy*step_size
        xi, yi = int(round(xb_new)), int(round(yb_new))
        if xi<0 or yi<0 or xi>=mask.shape[1] or yi>=mask.shape[0]:
            break
        if not mask[yi, xi]:
            break
        xb, yb = xb_new, yb_new
        steps += 1
    dist_backward = np.hypot(xb - x, yb - y)

    return dist_backward, dist_forward

# Compute distances for each peak
distances = {}
for (y, x), (eigval, eigvec) in directions.items():
    dx_vec, dy_vec = eigvec
    dist_b, dist_f = distance_to_edge(cleaned_mask, x, y, dx_vec, dy_vec)
    distances[(y, x)] = (dist_b, dist_f)

# -------------------------------
# 5️⃣ Plot symmetric double-headed arrows
# -------------------------------
plt.figure(figsize=(10, 8))
plt.imshow(J_par.T, cmap='seismic', origin='lower', vmin=-1.5, vmax=1.5)
plt.imshow(~cleaned_mask.T, cmap='gray', alpha=0.6, origin='lower')

for (y, x), (eigval, eigvec) in directions.items():
    dx_vec, dy_vec = eigvec
    dist_b, dist_f = distances[(y, x)]
    start = (x - dx_vec * dist_b, y - dy_vec * dist_b)
    end   = (x + dx_vec * dist_f, y + dy_vec * dist_f)
    arrow = FancyArrowPatch(start, end, arrowstyle='<->', color='red', 
                            mutation_scale=10, linewidth=2)
    plt.gca().add_patch(arrow)

plt.title(r'Maximum $J_\parallel$ + Steepest Descent Directions')
plt.xlabel(r'$x/d_p$')
plt.ylabel(r'$y/d_p$')
plt.tight_layout()
plt.show()

NameError: name 'max_coords' is not defined

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from scipy.ndimage import label

# ---------------------------------------------------------
# 1️⃣ Compute Hessian
# ---------------------------------------------------------
def compute_hessian(J, x, y, dx=1.0, dy=1.0):
    d2f_dx2 = (J[y, (x+1)%J.shape[1]] - 2*J[y, x] + J[y, (x-1)%J.shape[1]]) / dx**2
    d2f_dy2 = (J[(y+1)%J.shape[0], x] - 2*J[y, x] + J[(y-1)%J.shape[0], x]) / dy**2
    d2f_dxdy = (
        J[(y+1)%J.shape[0], (x+1)%J.shape[1]] - J[(y+1)%J.shape[0], (x-1)%J.shape[1]]
        - J[(y-1)%J.shape[0], (x+1)%J.shape[1]] + J[(y-1)%J.shape[0], (x-1)%J.shape[1]]
    ) / (4*dx*dy)
    return np.array([[d2f_dx2, d2f_dxdy],
                     [d2f_dxdy, d2f_dy2]])

# ---------------------------------------------------------
# 2️⃣ Compute steepest descent directions
# ---------------------------------------------------------
def steepest_descent_direction(J, max_coords, dx=1.0, dy=1.0):
    directions = {}
    for y, x in max_coords:
        H = compute_hessian(J, x, y, dx, dy)
        eigvals, eigvecs = np.linalg.eigh(H)
        idx = np.argmin(eigvals)  # most negative eigenvalue
        directions[(y, x)] = (eigvals[idx], eigvecs[:, idx])
    return directions

# ---------------------------------------------------------
# 3️⃣ Distance to region edges
# ---------------------------------------------------------
def distance_to_edge(mask, x, y, dx, dy, step_size=0.5, max_steps=1000):
    # Forward
    xf, yf = x, y
    steps = 0
    while steps < max_steps:
        xf_new = xf + dx*step_size
        yf_new = yf + dy*step_size
        xi, yi = int(round(xf_new)), int(round(yf_new))
        if xi < 0 or yi < 0 or xi >= mask.shape[1] or yi >= mask.shape[0]:
            break
        if not mask[yi, xi]:
            break
        xf, yf = xf_new, yf_new
        steps += 1
    dist_forward = np.hypot(xf - x, yf - y)

    # Backward
    xb, yb = x, y
    steps = 0
    while steps < max_steps:
        xb_new = xb - dx*step_size
        yb_new = yb - dy*step_size
        xi, yi = int(round(xb_new)), int(round(yb_new))
        if xi < 0 or yi < 0 or xi >= mask.shape[1] or yi >= mask.shape[0]:
            break
        if not mask[yi, xi]:
            break
        xb, yb = xb_new, yb_new
        steps += 1
    dist_backward = np.hypot(xb - x, yb - y)

    return dist_backward, dist_forward

# ---------------------------------------------------------
# 4️⃣ Main workflow
# ---------------------------------------------------------
directions = steepest_descent_direction(J_par, max_coords, dx=0.125, dy=0.125)

distances = {}
for (y, x), (eigval, eigvec) in directions.items():
    dx_vec, dy_vec = eigvec
    dist_b, dist_f = distance_to_edge(cleaned_mask, x, y, dx_vec, dy_vec)
    distances[(y, x)] = (dist_b, dist_f)

# Optional: compute total distances for histograms
total_distances = np.array([d[0]+d[1] for d in distances.values()])

# ---------------------------------------------------------
# 5️⃣ Plot symmetric arrows
# ---------------------------------------------------------
plt.figure(figsize=(10, 8))
plt.imshow(J_par.T, cmap='seismic', origin='lower', vmin=-1.5, vmax=1.5)
plt.imshow(~cleaned_mask.T, cmap='gray', alpha=0.6, origin='lower')

for (y, x), (eigval, eigvec) in directions.items():
    dx_vec, dy_vec = eigvec
    dist_b, dist_f = distances[(y, x)]
    start = (x - dx_vec * dist_b, y - dy_vec * dist_b)
    end   = (x + dx_vec * dist_f, y + dy_vec * dist_f)
    arrow = FancyArrowPatch(start, end, arrowstyle='<->', color='red', 
                            mutation_scale=10, linewidth=2)
    plt.gca().add_patch(arrow)

plt.title(r'Maximum $J_\parallel$ + Steepest Descent Directions')
plt.xlabel(r'$x/d_p$')
plt.ylabel(r'$y/d_p$')
plt.tight_layout()
plt.show()

NameError: name 'J_par' is not defined

In [4]:
import numpy as np
from scipy.ndimage import label

# ---------------------------------------------------------
# FUNCTIONS (Hessian + descent direction)
# ---------------------------------------------------------

def compute_hessian(J, x, y, dx=1.0, dy=1.0):
    """
    Compute 2D Hessian matrix at point (x,y) of array J using central differences.
    """
    d2f_dx2 = (J[y, (x+1)%J.shape[1]] - 2*J[y, x] + J[y, (x-1)%J.shape[1]]) / dx**2
    d2f_dy2 = (J[(y+1)%J.shape[0], x] - 2*J[y, x] + J[(y-1)%J.shape[0], x]) / dy**2
    d2f_dxdy = (
        J[(y+1)%J.shape[0], (x+1)%J.shape[1]] - J[(y+1)%J.shape[0], (x-1)%J.shape[1]]
        - J[(y-1)%J.shape[0], (x+1)%J.shape[1]] + J[(y-1)%J.shape[0], (x-1)%J.shape[1]]
    ) / (4*dx*dy)

    H = np.array([[d2f_dx2, d2f_dxdy],
                  [d2f_dxdy, d2f_dy2]])
    return H


def steepest_descent_direction(J, peaks, dx=1.0, dy=1.0):
    """
    For each peak, compute direction of steepest descent from Hessian.
    """
    directions = {}
    for (y, x) in peaks:
        H = compute_hessian(J, x, y, dx, dy)
        eigvals, eigvecs = np.linalg.eigh(H)  

        # Largest magnitude eigenvalue → steepest curvature
        idx = np.argmax(np.abs(eigvals))

        # Descent direction = eigenvector of *negative* curvature
        direction = -np.sign(eigvals[idx]) * eigvecs[:, idx]

        directions[(y, x)] = {
            "eigval": eigvals[idx],
            "eigvec": eigvecs[:, idx],
            "descent_dir": direction
        }

    return directions

# ---------------------------------------------------------
# FIND PEAKS USING CONNECTED REGIONS
# ---------------------------------------------------------

labeled_mask, num_features = label(cleaned_mask)
print("Regions found:", num_features)

peak_coords = []
peak_values = []

for i in range(1, num_features + 1):
    region_mask = (labeled_mask == i)
    region_vals = J_par[region_mask]

    if region_vals.size == 0:
        continue

    max_pos = np.argmax(np.abs(region_vals))
    extreme_value = region_vals[max_pos]
    peak_values.append(extreme_value)

    xs, ys = np.where(region_mask)
    peak_coords.append((xs[max_pos], ys[max_pos]))   # (y,x)

# print("Peak coordinates:", peak_coords)

# ---------------------------------------------------------
# COMPUTE STEEPEST DESCENT DIRECTIONS
# ---------------------------------------------------------

directions = steepest_descent_direction(J_par, peak_coords, 0.0625, 0.0625)

# print("\n--- Steepest descent directions ---")
# for coord, info in directions.items():
#     print(f"Peak at {coord}:")
#     print(f"  Eigenvalue: {info['eigval']}")
#     print(f"  Eigenvector: {info['eigvec']}")
#     print(f"  Descent direction: {info['descent_dir']}\n")

NameError: name 'cleaned_mask' is not defined

In [5]:
plt.figure(figsize=(10, 8))

plt.imshow(J_par.T, cmap='seismic', origin='lower', vmin=-1.5, vmax=1.5)
plt.imshow(~cleaned_mask.T, cmap='gray', alpha=0.6, origin='lower')

scale = 3.0

for (y, x), info in directions.items():

    # swap coordinates because the image is transposed
    plot_x = y
    plot_y = x

    # swap arrow direction too
    dy, dx = info["descent_dir"]
    dx_plot = dy * scale
    dy_plot = dx * scale

    # arrow
    plt.arrow(
        plot_x, plot_y,
        dx_plot, dy_plot,
        head_width=1.2,
        head_length=1.8,
        fc='yellow',
        ec='yellow',
        linewidth=2,
        length_includes_head=True
    )

plt.title("Steepest Descent Arrows (Corrected for Transpose)")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

NameError: name 'J_par' is not defined

<Figure size 1000x800 with 0 Axes>

In [6]:
from matplotlib.patches import FancyArrowPatch

plt.figure(figsize=(10, 8))

plt.imshow(J_par.T, cmap='seismic',
           origin='lower', vmin=-1.5, vmax=1.5)

plt.imshow(~cleaned_mask.T, cmap='gray', alpha=0.6, origin='lower')

factor = 100.0  # scale of arrows; adjust visually

ax = plt.gca()

for (y, x), info in directions.items():

    # Descent direction (dy, dx)
    dy, dx = info["descent_dir"]

    # ---- Fix coordinate swap because image is transposed ----
    # J_par.T means:
    #   plotted_x = y
    #   plotted_y = x
    px = y
    py = x

    # Swap direction components too
    dx_plot = dy
    dy_plot = dx

    # Normalize direction to get a stable arrow size
    norm = np.hypot(dx_plot, dy_plot)
    if norm == 0:
        continue

    dx_unit = dx_plot / norm
    dy_unit = dy_plot / norm

    # Compute symmetric endpoints
    start = (px - 0.5 * factor * dx_unit,
             py - 0.5 * factor * dy_unit)
    end   = (px + 0.5 * factor * dx_unit,
             py + 0.5 * factor * dy_unit)

    arrow = FancyArrowPatch(
        start, end,
        arrowstyle="<->",      # double-headed
        color="red",
        mutation_scale=15,     # arrowhead size
        linewidth=2,
    )
    ax.add_patch(arrow)

plt.title(r'Maximum $J_\parallel$ Per Connected Region + Steepest Descent Directions')
plt.xlabel(r'$x/d_p$')
plt.ylabel(r'$y/d_p$')

plt.tight_layout()
plt.savefig("Jpar_maxima_with_symmetric_descent_vectors.png", dpi=300)
plt.show()

NameError: name 'J_par' is not defined

<Figure size 1000x800 with 0 Axes>

In [7]:
import numpy as np
import matplotlib.pyplot as plt

def distance_to_boundary(cleaned_mask, start_y, start_x, dy, dx, step=0.2, max_steps=5000):
    """
    March from a point in a normalized direction (dy, dx)
    until exiting the connected region defined by cleaned_mask.
    Returns the traveled distance.
    """

    rows, cols = cleaned_mask.shape
    y, x = float(start_y), float(start_x)

    for i in range(max_steps):
        # step forward
        y += dy * step
        x += dx * step

        # stop if outside array
        if x < 0 or x >= cols or y < 0 or y >= rows:
            return i * step
        
        # stop if leaving region
        if not cleaned_mask[int(y), int(x)]:
            return i * step

    return max_steps * step  # fallback (very unlikely)


# ---------------------------------------------------------
# COMPUTE TOTAL BOUNDARY DISTANCE FOR EACH MAXIMUM
# ---------------------------------------------------------

total_distances1 = []

for (y, x), info in directions.items():

    dy, dx = info["descent_dir"]

    # normalize direction
    norm = np.hypot(dx, dy)
    if norm == 0:
        continue

    dy_u = dy / norm
    dx_u = dx / norm

    # forward (+ direction)
    d_forward = distance_to_boundary(cleaned_mask, y, x, dy_u, dx_u)

    # backward (- direction)
    d_backward = distance_to_boundary(cleaned_mask, y, x, -dy_u, -dx_u)

    total_distances1.append(d_forward + d_backward)


# ---------------------------------------------------------
# HISTOGRAM OF TOTAL DISTANCES
# ---------------------------------------------------------

plt.figure(figsize=(8,6))
plt.hist(total_distances1, bins=20, edgecolor="black")
plt.xlabel("Total distance across connected region (along descent direction)")
plt.ylabel("Count")
plt.title("Histogram of Total Descent-Span Distance per Maximum")
plt.tight_layout()
plt.show()

NameError: name 'directions' is not defined

In [8]:
plt.figure(figsize=(8,6))

plt.violinplot([total_distances, total_distances1, thicknesses],
               showmeans=True, showmedians=True)

plt.xticks([1,2,3], ["total_distance", "total_distance1", "thicknesses"])
plt.ylabel("Value")
plt.title("Violin Plot Comparison")
plt.tight_layout()
plt.show()

NameError: name 'total_distances' is not defined

<Figure size 800x600 with 0 Axes>

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

plt.hist(total_distances, bins=20, alpha=0.5, label="total_distance", edgecolor='black', density=True)
plt.hist(total_distances1, bins=20, alpha=0.5, label="total_distance1", edgecolor='black', density=True)
# plt.hist(thicknesses, bins=20, alpha=0.5, label="thicknesses", edgecolor='black', density=True)

plt.xlabel("Value")
plt.ylabel("Normalized Count")
plt.title("Comparison of Total Distances and Thicknesses (Normalized)")
plt.legend()
plt.tight_layout()
plt.show()